# 개인(근수) 통합 정리용 Notebook

- 프로젝트명: [금융] 젠트리피케이션 위험도 분석 시스템 구축 및 상생 체계 제안
- 분석 대상: 성남시 원도심 (수정구 41133 + 중원구 41131), 분당구 41135는 비교 대조군
- 담당자명: 근수
- 담당 파트: 가맹점 / 매출 / 공시지가
- 작성일: 2026-04-29
- 이 노트북의 목적: 근수 파트의 1차 전처리 결과, 실제 산출물, EDA 완료/미완료 상태를 한곳에 정리하고 최종 분석에 투입할 변수 세트를 확정
- 최종 산출물 한 줄 요약: 행정동(admi_cty_no) × 분기 단위 통합 변수표를 만들되, 현재 확정 산출물은 가맹점 EDA 지표(`open_rate`, `close_rate`, `franchise_ratio`, `industry_diversity_H`, `survival_rate_1y`)이며 `sales_amt`, `avg_land_price`, `land_price_growth`는 후속 EDA/집계 후 결합


## 1. 작업 개요

- 담당 데이터/업무: 가맹점(카드사 가맹점 통합), 매출, 공시지가 원본 -> 행정동×분기 단위 위험도 지표화
- 왜 필요한지: 본 프로젝트의 4대 핵심 지표(개·폐업률 / 프랜차이즈 침투율 / 업종 다양성 / 신생 점포 잔존율)와 부동산 압력(공시지가), 매출 활력 축이 최종 위험도 변수에 포함되기 때문
- 분석 범위: 2023-01 ~ 2025-12 (36개월), 성남시 50개 행정동, 시군구 3개(41131 중원·41133 수정·41135 분당), 블록코드 5,442개
- 분석 단위: 원도심(41131+41133) vs 비교 대조군(41135 분당구) 양극화 분석 구도
- 최종적으로 남길 파일/변수/표:
  - 전처리 완료 데이터: 전처리완료_카드(가맹점).csv, 전처리완료_부동산(공시지가).csv, 전처리완료_카드(매출).csv
  - EDA 산출 완료: 가맹점 4개 지표 + 종합 단계(탐색용 5단계)
  - 후속 산출 필요: 매출 EDA, 공시지가 EDA, 행정동×분기 통합 변수표
  - 최종 변수 연결표 1개


## 2. 삭제/백업/최종본 노트북 정리

현재 `Geunsu` 폴더 기준으로 실제 존재하는 노트북과 작성 필요 항목을 구분합니다.

| 파일명 | 현재 역할 | 유지 여부 | 비고 |
|---|---|---|---|
| 1차_전처리_가맹점.ipynb | 가맹점 1차 전처리 | 유지 | 완료, `전처리완료_카드(가맹점).csv` 산출 |
| 1차_전처리_매출.ipynb | 매출 1차 전처리 | 유지 | 완료, 최종 87,263,128행. 저장 경로는 `E:\전처리완료_카드(매출).csv` |
| 1차_전처리_공시지가.ipynb | 공시지가 1차 전처리 | 유지 | 완료, `전처리완료_부동산(공시지가).csv` 산출 |
| EDA_가맹점.ipynb | 전처리된 가맹점 데이터 탐색 | 유지 | 완료, 4대 핵심 지표 + 탐색용 5단계 분류 산출 |
| EDA_매출.ipynb | 매출 EDA | 작성 필요 | 은비님 담당 |
| EDA_공시지가.ipynb | 공시지가 EDA | 작성 필요 | 지륜님 담당 |
| 개인_통합정리_템플릿_근수.ipynb | 근수 파트 통합 정리 | 유지 | 실제 파일/산출물 상태 반영용 |


## 3. 문제 데이터 정리 섹션

1차 전처리 과정에서 실제로 발견·처리했거나 후속 처리가 필요한 이슈 기록입니다.

| 데이터셋 | 컬럼명 | 문제 유형 | 처리 방식 | 이유 | 확인 상태 |
|---|---|---|---|---|---|
| 가맹점 | ta_ym | int64 형식 (YYYYMM) | `pd.to_datetime(format='%Y%m')` 변환 | 분기 집계·시계열 분석을 위한 datetime 통일 | 완료 (range 2023-01 ~ 2025-12) |
| 가맹점 | cty_rgn_no / admi_cty_no / blk_cd | int64 -> 코드성 데이터 | `astype(str)` 변환 | 앞자리 0 보존, 조인 키 일관성 | 완료 |
| 가맹점 | sale | NULL 387,860건 (30억 초과 또는 국세청 미등록) | `'E'` 구간으로 채움 | 데이터 정의서상 의미 있는 결측 -> 신규 등급으로 분리 보존 | 완료 |
| 가맹점 | mm_cnt | 최댓값 134,368개월(약 1만년) — 비현실적 | 제거하지 않고 보존 (블록 단위 누적값으로 추정) | 동일 블록·업종 시계열에서 일관 추세 확인됨 | 검토 완료, 분석 시 행정동 단위 집계로 희석 |
| 가맹점 | open_cnt > mer_cnt | 11건 (mer_cnt=0인데 open_cnt>=1) | 보존 | 명세서: 신규 등록은 가맹점주 변경 재신청 포함 -> 정의상 가능 | 완료 |
| 가맹점 | stop_cnt > mer_cnt | 17건 | 보존 | 휴업 시점과 가맹점 수 집계 시점 차이 가능 | 완료 |
| 가맹점 | close_cnt > mer_cnt | 6,203건 | 분기 합산 후 비율 산출, 분모 안정성 필터 적용 | 월별 블록·업종 단위에서는 폐업 수가 현재 가맹점 수를 넘을 수 있음 | EDA 반영, 추가 검토 가능 |
| 가맹점 | 키컬럼(8개) 중복 | 0건 | — | 키 무결성 확인 완료 | 완료 |



## 4. 처리 기준 문서화 + 근수 전용 정리 포인트

### 처리 원칙
- 원본 유지: 원본 파일은 수정하지 않고 `전처리완료_*.csv` 형태로 별도 생성
- 결측 처리: 원본의 의미 있는 결측(예: `sale=NULL` -> 30억 초과)은 별도 카테고리로 보존(`'E'`)
- 비율 계산: 비율의 평균이 아닌 **합산 후 비율** 방식 채택 (Σopen_cnt / Σmer_cnt × 100)
- 분모 안정성 필터: 행정동×분기 mer_cnt 합 < 50 또는 open_11_cnt(t-12) < 5 인 조합은 비율 지표에서 제외
- 분기 단위: `ta_ym.dt.to_period("Q")` 변환 후 집계

### 공시지가 산/공원/비주거성 필지 처리 기준
- 현재 clean 파일 확인 컬럼: `법정동코드`, `법정동명`, `기준연도`, `기준월`, `기준년월`, `건물용도분류명`, `공시지가`
- 현재 clean 파일에는 `지목명` / `용도지역` 컬럼이 없어 산·공원·도로·하천 등 비상업 필지 제외 기준을 확정하기 어렵다.
- 우선 처리안: `건물용도분류명`으로 상업/비상업 후보를 분류하고, 판단이 불가능한 케이스는 원본 또는 sungju 공시지가 파트에서 지목·용도지역 정보를 보강한다.
- 판단이 어려운 케이스 기록 위치: `Geunsu/reference/공시지가_예외검토.csv` (현재 미작성)

### 최종 분석 변수 — 가맹점 핵심 지표 + 매출/부동산 압력 축

| 분석 축 | 대표 변수명 | 산식 | 단위 | 현재 상태 | 해석 방향 |
|---|---|---|---|---|---|
| 개·폐업 (활력) | `open_rate` | Σ open_cnt / Σ mer_cnt × 100 | 행정동 × 분기 | 가맹점 EDA 산출 | 높음 = 신규 진입 활발 |
| 개·폐업 (활력) | `close_rate` | Σ close_cnt / Σ mer_cnt × 100 | 행정동 × 분기 | 가맹점 EDA 산출 | 높음 = 퇴출 압력 강함 |
| 개·폐업 (활력) | `net_change` | open_rate - close_rate | 행정동 × 분기 | 가맹점 EDA 산출 | 음수 = 순감소(쇠퇴 신호) |
| 프랜차이즈 (체인화) | `franchise_ratio` | Σ fran_cnt / Σ mer_cnt × 100 | 행정동 × 분기 | 가맹점 EDA 산출 | 높음 = 프랜차이즈 침투 |
| 업종 다양성 | `industry_diversity_H` | H = -Σ(p_i × ln p_i), p_i = `card_tpbuz_nm_2`별 비율 | 행정동 × 분기 | 가맹점 EDA 산출 | 낮음 = 업종 획일화 |
| 신생 점포 잔존 | `survival_rate_1y` | open_12_23_cnt(t) / open_11_cnt(t-12) × 100 | 행정동 × 분기 (2024 Q1~) | 가맹점 EDA 산출 | 낮음 = 진입비용 충격 |

### 단계 분류 체계 정리
- 현재 `EDA_가맹점.ipynb` 산출물은 탐색용으로 **초기 / 주의 / 경계 / 위험 / 쇠퇴** 5단계를 사용한다.


## 5. 결과 요약표와 최종 변수 연결표

### 진행 상태 요약 (2026-04-29 기준)
| 데이터셋 | 1차 전처리 | EDA | 비고 |
|---|---|---|---|
| 가맹점 | 완료 | 완료 | `Geunsu/data/전처리완료_카드(가맹점).csv` 확인, 4대 핵심 지표 + 탐색용 5단계 분류 산출 |
| 매출 | 완료 | 필요 | 최종 87,263,128행. clean 파일은 `E:\전처리완료_카드(매출).csv` 저장으로 노트북 기록, 저장소 내 파일 미확인 |
| 공시지가 | 완료 | 필요 | `Geunsu/data/전처리완료_부동산(공시지가).csv` 확인, 행정동 매핑 및 `avg_land_price`/`land_price_growth` 산출 필요 |

### 가맹점 전처리 결과 요약표
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| 가맹점_정보(통합).csv | 전처리완료_카드(가맹점).csv | 2023-01 ~ 2025-12 | 2,130,071 | ta_ym, cty_rgn_no, admi_cty_no, blk_cd, card_tpbuz_cd, card_tpbuz_nm_1/2, sale, mm_cnt, mer_cnt, fran_cnt, open_cnt, stop_cnt, close_cnt, open_11~60_cnt | 4대 핵심 지표(개·폐업률 / 프랜차이즈 침투율 / 업종 다양성 / 신생 점포 잔존율) 산출 |

### 매출 전처리 결과 요약표
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| 매출_정보(통합).csv | 전처리완료_카드(매출).csv (`E:\` 저장 기록, 저장소 내 미확인) | 2023-01-01 ~ 2025-12-31 | 87,263,128 | ta_ymd, cty_rgn_no, admi_cty_no, card_tpbuz_cd, card_tpbuz_nm_1/2, hour, sex, age, day, amt, cnt | 행정동×분기 `sales_amt` 및 소비 활력 변수 산출 |

### 공시지가 전처리 결과 요약표
| 원본 파일 | 최종 파일명 | 기간 범위 | 총 행 수 | 핵심 컬럼 | 최종 사용 목적 |
|---|---|---|---|---|---|
| 성남시_공시지가_통합__202604241636.csv | 전처리완료_부동산(공시지가).csv | 2023 ~ 2025 추정(기준연도/기준월 기준 확인 필요) | 15,557 | 법정동코드, 법정동명, 기준연도, 기준월, 기준년월, 건물용도분류명, 공시지가 | 행정동 매핑 후 `avg_land_price`, `land_price_growth` 산출 |

### 핵심 데이터 분포 (가맹점 기준)
| 항목 | 값 |
|---|---|
| 시군구 분포 | 41135 분당 1,064,991 / 41133 수정 534,650 / 41131 중원 530,430 |
| 고유 행정동 수 | 50개 |
| 고유 블록코드 수 | 5,442개 |
| 업종 대분류 수 | 9개 (음식 · 소매/유통 · 생활서비스 ··· 공연/전시) |
| 업종 중분류 수 | 83개 |
| sale 분포 | A 1,069,467 / B 231,895 / C 214,109 / D 159,267 / E 387,860 / 신규 67,473 |

### 최종 변수 연결표 (원본 컬럼 -> 분석 변수)
| 데이터셋 | 원본 컬럼 | 최종 변수명 | 의미 | 사용 지표 | 산출 상태 |
|---|---|---|---|---|---|
| 가맹점 | mer_cnt | `mer_cnt` | 가맹점 수 (분모) | 모든 비율 지표 | 완료 |
| 가맹점 | open_cnt | `open_rate` (집계 후) | 신규 가맹점 비율 | 개업률 | 완료 |
| 가맹점 | close_cnt | `close_rate` (집계 후) | 폐업 가맹점 비율 | 폐업률 | 완료 |
| 가맹점 | fran_cnt | `franchise_ratio` (집계 후) | 프랜차이즈 비율 | 침투율 | 완료 |
| 가맹점 | card_tpbuz_nm_2 | `industry_diversity_H` (Shannon) | 업종 분포 엔트로피 | 업종 다양성 | 완료 |
| 가맹점 | open_11_cnt, open_12_23_cnt | `survival_rate_1y` | 1년 잔존율 (lag-12 비교) | 신생 점포 잔존 | 완료 |
| 가맹점 | sale | `sale` (E로 NULL 채움) | 매출 등급 | 대형 가맹점 비중 보조 | 완료 |

### 단계 분류 산출표 (현재 가맹점 EDA 스키마)
| 컬럼 | 타입 | 설명 |
|---|---|---|
| admi_cty_no | str(8) | 행정동 코드(최종 통합 시 필요) |
| 행정동명 | str | 매핑 결과 |
| 구명 | str | 분당구/수정구/중원구 |
| 개폐업률_단계 | str | 초기 / 주의 / 경계 / 위험 / 쇠퇴 |
| 프랜차이즈_단계 | str | 초기 / 주의 / 경계 / 위험 / 쇠퇴 |
| 업종다양성_단계 | str | 초기 / 주의 / 경계 / 위험 / 쇠퇴 |
| 잔존율_단계 | str | 초기 / 주의 / 경계 / 위험 / 쇠퇴 |
| 종합점수 | float | 4개 지표 단계 점수 평균(초기 0 ~ 쇠퇴 4) |
| 종합단계 | str | 초기 / 주의 / 경계 / 위험 / 쇠퇴 |


## 6. 최종 체크리스트 + 마지막 요약

### 체크리스트
- [x] 가맹점 1차 전처리 완료 (2,130,071행, 2023-01 ~ 2025-12)
- [x] 키 컬럼 8개 기준 중복 0건 확인
- [x] sale NULL 387,860건 -> 'E' 등급 보존 처리
- [x] 시군구 코드 3개(41131/41133/41135) · 행정동 50개 · 블록 5,442개 무결성 확인
- [x] 가맹점 EDA 완료 (4대 핵심 지표 + 탐색용 5단계 분류 + 종합단계)
- [x] 매출 1차 전처리 완료 (0매출/0건수 제거 후 87,263,128행)
- [ ] 매출 clean 파일 `Geunsu/data/전처리완료_카드(매출).csv` 저장 확인
- [ ] 매출 EDA 완료 (행정동×분기 `sales_amt` 추이, 업종별 매출 변화)
- [x] 공시지가 1차 전처리 완료 (`전처리완료_부동산(공시지가).csv`, 15,557행)
- [ ] 공시지가 행정동 매핑 기준 확정
- [ ] 공시지가 산/공원/비상업 필지 제외 기준 확정 및 적용
- [ ] 공시지가 EDA 완료 (`avg_land_price`, `land_price_growth` 산출)
- [ ] 행정동×분기 통합 변수표 1개 산출 (`open_rate`, `close_rate`, `franchise_ratio`, `industry_diversity_H`, `survival_rate_1y`, `avg_land_price`, `land_price_growth`, `sales_amt`)
- [ ] clean 파일 3종 `Geunsu/data/` 또는 팀 공통 경로 저장 완료
- [ ] 노트북 중복 정리(전처리 3종 + EDA 3종 + 통합정리 1종) 완료
- [ ] 팀 최종 4단계 체계에 맞춰 가맹점 EDA의 `쇠퇴` 단계 재매핑 여부 결정

### 현재 진행 상태 (2026-04-29)
- **완료**: 가맹점 1차 전처리 + EDA, 매출 1차 전처리, 공시지가 1차 전처리
- **확인 필요**: 매출 clean CSV의 저장소 내 위치, 공시지가 행정동 매핑 기준
- **다음 작업**: 매출·공시지가 EDA 작성 후 행정동×분기 통합 변수표 산출

### 팀 공유용 5줄 요약
1. 성남시 가맹점 데이터 213만건을 행정동(50개) × 분기 단위로 정제하여 4대 핵심 지표(개·폐업률 / 프랜차이즈 침투율 / 업종 다양성 H / 신생 점포 잔존율)를 산출했습니다.
2. `EDA_가맹점.ipynb`는 현재 탐색용 5단계(초기/주의/경계/위험/쇠퇴)를 사용하므로, 팀 최종 4단계(초기/주의/경계/위험)와 맞추려면 `쇠퇴` 단계 재매핑 기준이 필요합니다.
3. 매출 데이터는 0매출/0건수 제거 후 87,263,128행으로 1차 전처리되었고, 저장소 내 clean 파일 위치 확인 및 행정동×분기 `sales_amt` EDA가 남아 있습니다.
4. 공시지가는 15,557행 clean 파일이 확인되었지만, 행정동 매핑과 `avg_land_price`, `land_price_growth` 산출은 아직 필요합니다.
5. 최종 후속 산출물은 `전처리완료_카드(가맹점).csv` + `전처리완료_카드(매출).csv` + `전처리완료_부동산(공시지가).csv` + 행정동×분기 통합 변수표입니다.
